# Semantic Role Labeling v2 — Bug-Fixed Baselines + Predicate-Conditioned Transformer-CRF

Upgraded experimental pipeline comparing **Rule-Based v2**, **CRF v2 (LCA paths + aligned spaCy)**, and **Predicate-Conditioned Transformer + Constrained Linear-Chain CRF** across **PropBank-EWT (English)** and **PropBank-ES / AnCora (Spanish)**.

### Key v2 Upgrades over v1 (`Semantic Role Labeling.pdf`)
1. **Rule-Based v2:** Fixes missing `ARGM-`/`AM-` prefix on prepositional rules (previously 0.00 F1 on `ARGM-LOC`/`TMP`/`ADV`), adds UD `obl -> case` traversal for Spanish, adds `ARGM-MOD`, lexicalized `advmod`/`advcl`/`npadvmod` rules, `xcomp`/`conj` inherited subjects, and punctuation trimming.
2. **CRF v2:** Fixes token-drift misalignment (`Doc(vocab, words=w)` instead of `" ".join(w)`), adds true LCA directional dependency paths, `top_arg_dep` + governing preposition + subtree boundary cues, and fixes L-BFGS convergence speed (`min_freq=3`, `all_possible_transitions=False`).
3. **Transformer v2:** Predicate-conditioned word-level representation `[h_i; h_pred; h_i * h_pred; e_rel]` + pure-PyTorch **BIO-Constrained Linear-Chain CRF**, dynamic padding (`max_length=256`), differential LR (`backbone=2e-5`, `head+CRF=1e-3`), T4-safe mixed precision, dev-set Span F1 checkpointing + early stopping, and stronger backbones (`microsoft/deberta-v3-base` for EN, `PlanTL-GOB-ES/roberta-base-bne` / BETO for ES).

> **Resume-safe:** Results are persisted to Google Drive (`results_v2_full.json`) after **every** model finishes, so Colab disconnects never lose completed runs.


In [ ]:
# 1. Setup environment (Run on Colab T4 / L4 / A100 GPU)
!pip install -q transformers sentencepiece protobuf datasets seqeval sklearn-crfsuite "spacy[transformers]" tqdm matplotlib
!python -m spacy download en_core_web_trf -q
!python -m spacy download es_dep_news_trf -q


In [ ]:
import os, sys, json, csv, time
import torch
import spacy
from google.colab import drive

drive.mount('/content/drive')

# Clone or update your repo containing src/srl/
REPO_DIR = '/content/Semantic-Role-Labeling'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/yavuzselimuyaner/Semantic-Role-Labeling.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

PROPBANK_PATH    = '/content/drive/MyDrive/colab_upload/SRL_data/propbank_preprocessed.json'
PROPBANK_ES_PATH = '/content/drive/MyDrive/colab_upload/SRL_data/propbank_es_preprocessed.json'
OUT_DIR          = '/content/drive/MyDrive/colab_upload/SRL_data/results_v2'
os.makedirs(OUT_DIR, exist_ok=True)

RESULTS_JSON = os.path.join(OUT_DIR, 'results_v2_full.json')
if os.path.exists(RESULTS_JSON):
    with open(RESULTS_JSON, encoding='utf-8') as f:
        all_results = json.load(f)
    print('Loaded existing partial results:', {k: list(v.keys()) for k, v in all_results.items()})
else:
    all_results = {'propbank': {}, 'propbank_es': {}}

def save_checkpoint_results():
    with open(RESULTS_JSON, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)
    print(f'  [saved] {RESULTS_JSON}')

print('Device:', 'cuda (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'cpu')


In [ ]:
from srl.data import load_dataset, get_label_set, dataset_stats
from srl.metrics import compute_metrics, error_analysis, print_results, print_error_analysis
from srl.rule_based import RuleBasedSRL
from srl.crf_model import CRFModel, parse_aligned_docs
from srl.transformer_model import TransformerSRLModel

# Dataset sanity & span-length diagnostics
datasets_cfg = {
    'propbank': PROPBANK_PATH,
    'propbank_es': PROPBANK_ES_PATH,
}
loaded = {}
for name, path in datasets_cfg.items():
    tr, dv, te = load_dataset(path)
    loaded[name] = (tr, dv, te)
    st = dataset_stats(te, name=f'{name}_test')
    print(f'[{name}] Train={len(tr):,} Dev={len(dv):,} Test={len(te):,} | Labels={len(get_label_set(tr, dv, te))}')
    print('  Test stats:', st)


In [ ]:
# 2. Run Rule-Based v2 + CRF v2 (1-to-1 aligned spaCy TRF parse, shared docs)
nlp_models = {
    'propbank': spacy.load('en_core_web_trf', disable=['ner']),
    'propbank_es': spacy.load('es_dep_news_trf', disable=['ner']),
}

for ds_name in ['propbank', 'propbank_es']:
    need_rule = 'rule_based_v2' not in all_results[ds_name]
    need_crf  = 'crf_v2' not in all_results[ds_name]
    if not (need_rule or need_crf):
        print(f'Skipping {ds_name} rule+crf (already in results_v2_full.json)')
        continue

    train, dev, test = loaded[ds_name]
    gold_test = [inst['labels'] for inst in test]
    nlp = nlp_models[ds_name]

    print(f'\n==== Parsing aligned test docs for {ds_name} ====')
    test_docs = parse_aligned_docs(test, nlp, batch_size=256)

    if need_rule:
        t0 = time.time()
        rb = RuleBasedSRL(nlp=nlp, mode=ds_name)
        rb_preds = rb.predict_batch(test, docs=test_docs)
        rb_m = compute_metrics(gold_test, rb_preds)
        rb_ea = error_analysis(gold_test, rb_preds)
        print_results(f'Rule-Based v2 [{ds_name}]', rb_m)
        print_error_analysis(f'Rule-Based v2 [{ds_name}]', rb_ea)
        all_results[ds_name]['rule_based_v2'] = {**rb_m, 'error_analysis': rb_ea, 'time_s': round(time.time()-t0, 2)}
        save_checkpoint_results()

    if need_crf:
        t0 = time.time()
        print(f'Parsing aligned train docs for {ds_name} CRF...')
        train_docs = parse_aligned_docs(train, nlp, batch_size=256)
        crf = CRFModel(max_iterations=200, c1=0.1, c2=0.1, min_freq=3, all_possible_transitions=False, verbose=True)
        crf.fit(train, nlp, docs=train_docs)
        crf_preds = crf.predict(test, nlp, docs=test_docs)
        crf_m = compute_metrics(gold_test, crf_preds)
        crf_ea = error_analysis(gold_test, crf_preds)
        print_results(f'CRF v2 [{ds_name}]', crf_m)
        print_error_analysis(f'CRF v2 [{ds_name}]', crf_ea)
        all_results[ds_name]['crf_v2'] = {**crf_m, 'error_analysis': crf_ea, 'time_s': round(time.time()-t0, 2)}
        save_checkpoint_results()

del nlp_models
torch.cuda.empty_cache()


In [ ]:
# 3. Run Predicate-Conditioned Transformer + Constrained Linear-Chain CRF
# Controlled ablation + SOTA backbone comparison:
TRANSFORMER_RUNS = [
    # English: controlled BERT+CRF ablation vs DeBERTa-v3+CRF
    ('propbank',    'bert-base-cased',                        'bert_base_crf',    6, 32, 3e-5),
    ('propbank',    'microsoft/deberta-v3-base',              'deberta_v3_crf',   6, 16, 2e-5),
    # Spanish: BETO+CRF vs MarIA RoBERTa-BNE+CRF
    ('propbank_es', 'dccuchile/bert-base-spanish-wwm-cased',  'beto_crf',         6, 32, 3e-5),
    ('propbank_es', 'PlanTL-GOB-ES/roberta-base-bne',         'roberta_bne_crf',  6, 32, 2e-5),
]

for ds_name, model_name, run_tag, epochs, bsz, lr in TRANSFORMER_RUNS:
    if run_tag in all_results[ds_name]:
        print(f'Skipping {ds_name}/{run_tag} (already completed, F1={all_results[ds_name][run_tag]["f1"]:.4f})')
        continue

    train, dev, test = loaded[ds_name]
    gold_test = [inst['labels'] for inst in test]
    label_set = get_label_set(train, dev, test)

    t0 = time.time()
    ckpt_path = os.path.join(OUT_DIR, f'{ds_name}_{run_tag}.pt')
    model = TransformerSRLModel(
        label_set=label_set,
        model_name=model_name,
        language=ds_name,
        max_length=256,
        batch_size=bsz,
        epochs=epochs,
        backbone_lr=lr,
        head_lr=1e-3,
        use_crf=True,
        enforce_bio=True,
        patience=3,
    )
    history = model.fit(train, dev_instances=dev, save_path=ckpt_path)
    preds = model.predict(test)
    m = compute_metrics(gold_test, preds)
    ea = error_analysis(gold_test, preds)
    print_results(f'{run_tag} [{ds_name}]', m)
    print_error_analysis(f'{run_tag} [{ds_name}]', ea)
    all_results[ds_name][run_tag] = {
        **m, 'model_name': model_name, 'history': history,
        'error_analysis': ea, 'time_s': round(time.time()-t0, 2),
    }
    save_checkpoint_results()
    del model
    torch.cuda.empty_cache()


In [ ]:
# 4. Export final summary CSV & print v1 vs v2 comparison
V1_BASELINE = {
    'propbank':    {'rule_based_v1': 0.3397, 'crf_v1': 0.5453, 'bert_v1': 0.7343},
    'propbank_es': {'rule_based_v1': 0.1438, 'crf_v1': 0.2952, 'mbert_v1': 0.4594},
}

rows = []
print('\n' + '=' * 78)
print(f'  {"Dataset":<12} {"Method":<18} {"Prec":>8} {"Rec":>8} {"F1":>8} {"BadBIO":>7} {"Time(s)":>8}')
print('-' * 78)
for ds_name, methods in all_results.items():
    for method, r in methods.items():
        print(f'  {ds_name:<12} {method:<18} {r["precision"]:>8.4f} {r["recall"]:>8.4f} {r["f1"]:>8.4f} {r.get("illegal_bio",0):>7} {r.get("time_s",0):>8.1f}')
        rows.append({
            'dataset': ds_name, 'method': method,
            'precision': r['precision'], 'recall': r['recall'], 'f1': r['f1'],
            'illegal_bio': r.get('illegal_bio', 0), 'time_s': r.get('time_s', 0),
        })
print('=' * 78)

csv_path = os.path.join(OUT_DIR, 'results_v2_summary.csv')
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)

print('Saved:', csv_path, 'and', RESULTS_JSON)
